# injector_demo

injector - 模块描述

作者: huangyoubin
创建日期: 2026/7/30 12:27

知识点讲解：Injector 依赖注入框架

1. 依赖注入（Dependency Injection，DI）简介
   - 一种设计模式，将对象的依赖关系外部化
   - 对象不主动创建依赖，而是由外部容器注入
   - 核心目的：降低耦合、提升可测试性、便于扩展

2. 为什么需要依赖注入
   - 不使用 DI：类内部 new 依赖对象，硬编码耦合
   - 使用 DI：依赖由容器管理，可灵活替换实现
   - 单元测试时可以注入 Mock 对象
   - 修改依赖实现无需改动业务代码

3. Injector 库简介
   - Python 的依赖注入框架，类似 Java 的 Guice
   - 基于类型注解（type hints）自动解析依赖
   - 支持模块化配置、作用域管理、父子容器

4. @inject 装饰器详解
   - 标记类或函数需要依赖注入
   - Injector 会读取 __init__ 的类型注解
   - 自动解析并注入对应类型的实例
   - 本示例中 B 依赖 A，通过类型注解 a: A 声明

5. Injector 容器
   - Injector(modules=...)：创建注入容器
   - get(类型)：从容器获取指定类型的实例
   - 自动递归解析依赖链
   - 默认单例作用域（同一容器多次 get 返回同一实例）

6. binder 与绑定配置
   - configure(binder) 函数用于配置依赖绑定
   - binder.bind(接口/类型, to=具体实现)
   - 绑定方式：
     * to=实例：绑定到具体实例（本示例）
     * to=类：绑定到类（由容器实例化）
     * to=工厂函数：绑定到工厂方法

7. 为什么需要手动绑定 A
   - A 的 __init__ 需要 name 参数（运行时参数）
   - Injector 无法自动推断 name 的值
   - 必须通过 binder.bind(A, to=A(name="llmops")) 提供实例
   - 对比：B 只依赖类型 A，可以自动注入

8. 父子容器（Parent-Child Injector）
   - Injector(modules=..., parent=父容器)
   - 子容器优先使用自己的绑定
   - 找不到时向父容器查找
   - 应用场景：
     * 全局配置在父容器，请求级配置在子容器
     * 多租户场景：不同租户不同配置
     * 测试时覆盖部分依赖

9. 本示例的绑定覆盖分析
   - 父容器绑定：A(name="llmops")
   - 子容器绑定：A(name="child-llmops")
   - injector.get(B) 时，B 依赖的 A 从子容器获取
   - 输出结果：name: child-llmops（子容器覆盖父容器）

10. 在 LLMOps 项目中的应用
    - Flask 应用中管理 Service、Repository 等组件
    - 数据库连接、LLM 客户端等资源的统一管理
    - 配置对象的注入
    - 便于单元测试时替换真实依赖

11. 作用域（Scope）
    - NoScope（默认）：每次 get 创建新实例
    - SingletonScope：容器内单例
    - ThreadLocalScope：线程内单例
    - RequestScope：请求内单例（Web 应用）

12. 最佳实践
    - 优先使用构造函数注入（而非属性注入）
    - 使用类型注解明确依赖关系
    - 需要运行时参数的依赖通过 binder 手动绑定
    - 模块化组织绑定配置（按功能拆分 configure 函数）
    - 避免在类内部直接 new 依赖对象
    - 注意避免循环依赖

13. 常见陷阱
    - 类型注解缺失导致注入失败
    - 循环依赖导致递归错误
    - 忘记加 @inject 装饰器
    - 局部模块名与第三方包同名导致导入冲突
      （本文件命名为 injector_demo.py 而非 injector.py）


In [ ]:
from injector import Injector, inject

In [ ]:
class A:
    def __init__(self, name: str):
        self.name = name

In [ ]:
@inject
class B:
    def __init__(self, a: A):
        self.a = a
    def print(self):
        print(f"name: {self.a.name}")

In [ ]:
def configure(binder):
    """为需要运行时参数的依赖提供具体实例。"""
    binder.bind(A, to=A(name="llmops"))

In [ ]:
def child_configure(binder):
    binder.bind(A, to=A(name="child-llmops"))

In [ ]:
if __name__ == "__main__":
    parent_injector = Injector(modules=configure)
    injector = Injector(modules=child_configure, parent=parent_injector)
    b = injector.get(B)
    b.print()